# 07 withColumn

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Add, replace and derive columns with <code>withColumn</code>, <code>withColumns</code> and <code>lit</code>, rename columns with <code>withColumnRenamed</code>, and know why calling <code>withColumn</code> in a long loop is an anti-pattern.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Almost every transformation step adds or fixes a column: a cleaned name, a converted amount, an audit timestamp, a derived flag. <code>withColumn</code> is how you do it in PySpark. The "<code>withColumn</code> in a loop" performance problem is a well-known interview topic.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 How withColumn works</b><br>
<code>df.withColumn("name", expression)</code> returns a new DataFrame with every existing column plus the new one.
<ul><li>If a column called <code>name</code> already exists, it is <b>replaced</b> (same position)</li><li><code>F.lit(value)</code> creates a constant column</li><li><code>withColumns({...})</code> adds or replaces several columns in one call (Spark 3.3+)</li><li><code>withColumnRenamed(old, new)</code> renames, and <code>withColumnsRenamed({...})</code> renames several</li><li>Each call adds a projection to the plan. Hundreds of calls make the plan huge and slow to analyze</li></ul>
</div>

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A payments pipeline receives amounts in minor units (cents) and in several currencies. The Silver step adds <code>amount</code> (cents / 100), <code>amount_usd</code> (amount × FX rate), <code>is_high_value</code> (amount_usd > 10,000), <code>ingest_ts</code> (current timestamp) and <code>source_file</code> (a literal). That is five derived columns on every row, which one <code>withColumns</code> call handles cleanly.
</div>

## Syntax

```python
from pyspark.sql import functions as F

df = df.withColumn("bonus", F.col("salary") * 0.1)          # add
df = df.withColumn("salary", F.col("salary").cast("double")) # replace
df = df.withColumn("country", F.lit("IN"))                  # constant
df = df.withColumns({"a": F.lit(1), "b": F.col("x") + 1})   # several at once
df = df.withColumnRenamed("emp_name", "employee_name")      # rename
df = df.withColumnsRenamed({"a": "alpha", "b": "beta"})     # rename several
```

## Setup: sample data

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates an orders DataFrame with amounts in cents and currency codes.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A 5-row table: <code>order_id</code>, <code>customer</code>, <code>amount_cents</code>, <code>currency</code>, <code>order_ts</code>.
</div>

In [0]:
from pyspark.sql import functions as F

orders = spark.createDataFrame(
    [
        (1, " amal ", 250000, "INR", "2024-05-01 10:15:00"),
        (2, "SENTHIL", 1999900, "USD", "2024-05-01 11:00:00"),
        (3, "john", 45000, "GBP", "2024-05-02 09:30:00"),
        (4, "Vivek", 120000, "AED", "2024-05-02 18:45:00"),
        (5, "priya", 5500, "USD", "2024-05-03 07:05:00"),
    ],
    "order_id INT, customer STRING, amount_cents BIGINT, currency STRING, order_ts STRING",
)
display(orders)

## 1. Add a derived column

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Adds <code>amount</code> by converting cents to the main currency unit.<br><br>
<b>Why it matters</b><br>Deriving a column from others is the most common use of <code>withColumn</code>.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The original 5 columns plus <code>amount</code> (for example <code>2500.0</code> for order 1).
</div>

In [0]:
orders_amt = orders.withColumn("amount", F.col("amount_cents") / 100)
orders_amt.show()

## 2. Replace an existing column

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Re-uses an existing column name to clean <code>customer</code> and convert <code>order_ts</code> to a real timestamp.<br><br>
<b>Why it matters</b><br>Replacing in place keeps the column order and avoids leftover raw columns. It is the standard way to clean and cast.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>customer</code> is trimmed and in title case (<code>Amal</code>, <code>Senthil</code>, ...), and <code>order_ts</code> shows as <code>timestamp</code> in the schema. Same number of columns as before.
</div>

In [0]:
cleaned = (
    orders_amt
    .withColumn("customer", F.initcap(F.trim(F.col("customer"))))
    .withColumn("order_ts", F.to_timestamp("order_ts"))
)
cleaned.show()
cleaned.printSchema()

## 3. Constants with `lit` and audit columns

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Adds a constant source name, the load timestamp, and a lookup-based FX rate.<br><br>
<b>Why it matters</b><br>Audit columns (where did this row come from and when was it loaded?) are required in Bronze and Silver tables. <code>F.lit</code> turns a Python value into a column. Without it you get an error.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
New columns <code>source_system</code> = <code>shop_api</code>, <code>loaded_at</code> = current timestamp, <code>fx_to_usd</code> (1.0 for USD, 0.012 for INR, and so on) and <code>amount_usd</code>.
</div>

In [0]:
# A small mapping from currency to USD rate, turned into a map column with create_map.
from itertools import chain

fx = {"USD": 1.0, "INR": 0.012, "GBP": 1.27, "AED": 0.27}
fx_map = F.create_map([F.lit(x) for x in chain(*fx.items())])

enriched = (
    cleaned
    .withColumn("source_system", F.lit("shop_api"))
    .withColumn("loaded_at", F.current_timestamp())
    .withColumn("fx_to_usd", fx_map[F.col("currency")])
    .withColumn("amount_usd", F.round(F.col("amount") * F.col("fx_to_usd"), 2))
)
enriched.show(truncate=False)

In [0]:
# Without lit(), a plain Python value is not a Column, so PySpark raises an error.
try:
    cleaned.withColumn("source_system", "shop_api")
except Exception as e:
    print(type(e).__name__, "-", str(e).split("\n")[0][:120])

## 4. Several columns at once with `withColumns`

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Adds or replaces several columns in one call by passing a dictionary.<br><br>
<b>Why it matters</b><br>It is shorter, and it produces a single projection in the plan instead of one per column.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Watch out</b><br>
Columns in the same <code>withColumns</code> call can't reference each other. <code>amount_usd</code> can't use <code>amount</code> if <code>amount</code> is created in the same dictionary. Chain two calls when one new column depends on another.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The cleaned orders plus <code>order_date</code>, <code>order_hour</code> and <code>is_high_value</code>.
</div>

In [0]:
features = enriched.withColumns({
    "order_date": F.to_date("order_ts"),
    "order_hour": F.hour("order_ts"),
    "is_high_value": F.col("amount_usd") > 1000,
})
features.select("order_id", "amount_usd", "order_date", "order_hour", "is_high_value").show()

## 5. Renaming columns

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Renames one column with <code>withColumnRenamed</code>, then several with <code>withColumnsRenamed</code>.<br><br>
<b>Why it matters</b><br>Renaming to business-friendly names happens at the Gold layer, and to consistent snake_case at Silver.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Watch out</b><br>
<code>withColumnRenamed</code> <b>does nothing</b> (no error) if the old column doesn't exist. A typo leaves your column unchanged without any warning.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>customer</code> becomes <code>customer_name</code>, then <code>amount_usd</code> and <code>order_ts</code> become <code>order_value_usd</code> and <code>ordered_at</code>. The typo example still shows <code>customer_name</code>.
</div>

In [0]:
renamed = features.withColumnRenamed("customer", "customer_name")
renamed = renamed.withColumnsRenamed({"amount_usd": "order_value_usd", "order_ts": "ordered_at"})
print(renamed.columns)

# Silent no-op: 'custmer_name' doesn't exist, so nothing changes and no error is raised.
print(renamed.withColumnRenamed("custmer_name", "client").columns[:3])

## 6. Anti-pattern: `withColumn` in a loop

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Adds 100 columns with one <code>select</code>, then the same 100 in a Python loop with <code>withColumn</code>, and compares the time to build the plan.<br><br>
<b>Why it matters</b><br>Each <code>withColumn</code> creates a new DataFrame and the analyzer re-checks the growing plan every time. With many columns this becomes very slow, and can even fail with a <code>StackOverflowError</code> or an analyzer iteration limit. The Spark docs explicitly warn about it.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The single <code>select</code> builds 101 columns almost instantly. The loop is much slower to <b>plan</b> (no data is processed yet), and on Spark Connect / serverless it may even fail with an analyzer error such as <code>Max iterations (100) reached for batch Resolution</code>. Exact behavior and times vary, so write down what you see.
</div>

In [0]:
import time

base = spark.range(10)

# The efficient way: one select with a list of expressions.
start = time.time()
select_df = base.select("*", *[(F.col("id") * i).alias(f"c{i}") for i in range(100)])
print("single select:", len(select_df.columns), "columns in", round(time.time() - start, 2), "s")

# The anti-pattern: 100 withColumn calls, each wrapping the previous plan.
start = time.time()
try:
    loop_df = base
    for i in range(100):
        loop_df = loop_df.withColumn(f"c{i}", F.col("id") * i)
    loop_df.columns  # forces analysis of the whole plan
    print("withColumn loop:", len(loop_df.columns), "columns in", round(time.time() - start, 2), "s")
except Exception as e:
    print("withColumn loop FAILED after", round(time.time() - start, 2), "s:", str(e).splitlines()[0][:160])

## Under the hood

```
df.withColumn("a", expr1).withColumn("b", expr2).withColumn("c", expr3)
        |
        v
 Analyzed plan:   Project(c) <- Project(b) <- Project(a) <- source   (one per call)
        |
        v
 Optimized plan:  projections are merged where possible (rule: CollapseProject)
        |
        v
 Physical plan:   *(1) Project ... all in ONE code-generated stage, one pass over each partition
```

<span style="background:#2F6FED;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Transformation</span> `withColumn` adds a Project, nothing runs yet.

<span style="background:#1E9E5A;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Narrow</span> new columns are computed row by row, no shuffle.

Run the cell below. Column `b` re-uses the computed column `a`, so the optimizer **keeps two Projects** instead of merging them (merging would compute `id + 1` twice). Both are marked `*(1)`, meaning they run in the same whole-stage code-generated function, so the data is still read only once. The real cost of long `withColumn` chains is **planning** on the driver, not execution.

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Two <code>Project</code> nodes in the optimized and physical plans, both prefixed with <code>*(1)</code> in the physical plan.
</div>

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: <code>[NOT_COLUMN_OR_STR]</code> or <code>col should be Column</code></b><br>
You passed a Python value directly. Wrap constants in <code>F.lit(...)</code>.<br><br>
<b>⛔ Problem: rename silently did nothing</b><br><code>withColumnRenamed</code> ignores missing columns. Check <code>df.columns</code> after renaming, or assert the name exists first.<br><br>
<b>⛔ Problem: the driver is slow before any job starts</b><br>Many <code>withColumn</code> calls in a loop. Build a list of expressions and use one <code>select</code> or <code>withColumns</code>.<br><br>
<b>⛔ Problem: new column is all null</b><br>The expression references a null column, or a cast failed under <code>try_cast</code>, or the map lookup key is missing. Inspect the inputs.<br><br>
**⛔ Problem: <code>withColumns</code> can't find a column it just created**<br>Columns in the same dictionary can't reference each other. Chain two calls.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. What does <code>withColumn</code> do?</b><br>
It returns a new DataFrame with a column added, or replaced if the name already exists. It is a lazy, narrow transformation that adds a Project node to the plan.<br><br>

<b>🎤 2. What is <code>F.lit</code> and why is it needed?</b><br>
It wraps a Python constant into a Column. DataFrame methods expect Column expressions, so a bare string or number must be wrapped.<br><br>

<b>🎤 3. Why is calling <code>withColumn</code> in a loop a problem?</b><br>
Each call creates a new DataFrame whose plan must be analyzed again, so the driver spends a long time planning and very large plans can overflow the stack. Use one <code>select</code> with a list of expressions, or <code>withColumns</code>.<br><br>

<b>🎤 4. What is the difference between <code>withColumn</code> and <code>select</code>?</b><br>
<code>withColumn</code> keeps all existing columns and adds or replaces one. <code>select</code> defines the whole output column list. Both produce a Project in the plan.<br><br>

<b>🎤 5. What happens if you rename a column that doesn't exist?</b><br>
Nothing. <code>withColumnRenamed</code> is a no-op for missing columns and raises no error, so typos can slip through.<br><br>

<b>🎤 6. How would you add an audit timestamp and source column to every row?</b><br>
<code>df.withColumns({"ingest_ts": F.current_timestamp(), "source": F.lit("crm")})</code>. On Databricks, file-based sources also expose <code>_metadata.file_path</code> for the source file name.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. Which code adds a column <code>load_date</code> containing today's date to every row?</b><br>
A. <code>df.withColumn("load_date", "current_date()")</code><br>
B. <code>df.withColumn("load_date", F.current_date())</code><br>
C. <code>df.withColumn(F.current_date(), "load_date")</code><br>
D. <code>df.withColumnRenamed("load_date", F.current_date())</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b> The second argument must be a Column expression. A passes a string, which is an error.</details><br><br>

<b>Q2. A team adds 300 derived columns with <code>withColumn</code> inside a loop and the notebook spends minutes before any Spark job starts. What is the best fix?</b><br>
A. Increase the number of executors<br>
B. Cache the DataFrame after every <code>withColumn</code><br>
C. Build all 300 expressions in a list and use a single <code>select</code><br>
D. Convert the DataFrame to pandas first<br>
<details><summary><b>Show answer</b></summary><b>C.</b> The delay is driver-side plan analysis, not execution, so more executors or caching won't help.</details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b><br>
Using <code>orders</code>:
<ol><li>Add <code>amount</code> (cents / 100) and replace <code>customer</code> with a trimmed, upper-case version</li><li>Add <code>channel</code> set to the constant <code>"web"</code></li><li>With one <code>withColumns</code> call, add <code>order_date</code> (date of <code>order_ts</code>) and <code>is_weekend</code> (true for Saturday or Sunday; hint: <code>F.dayofweek</code> returns 1 for Sunday and 7 for Saturday)</li><li>Rename <code>amount_cents</code> to <code>amount_minor_units</code></li><li>Add 20 columns <code>f0</code> ... <code>f19</code> (each <code>amount_cents * i</code>) the efficient way</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1-2. Derived, replaced and constant columns
p = (
    orders
    .withColumn("amount", F.col("amount_cents") / 100)
    .withColumn("customer", F.upper(F.trim("customer")))
    .withColumn("channel", F.lit("web"))
)

# 3. Two columns at once. dayofweek: 1 = Sunday, 7 = Saturday
p = p.withColumns({
    "order_date": F.to_date("order_ts"),
    "is_weekend": F.dayofweek(F.to_timestamp("order_ts")).isin(1, 7),
})

# 4. Rename
p = p.withColumnRenamed("amount_cents", "amount_minor_units")

# 5. Many columns in a single select instead of a loop
p = p.select("*", *[(F.col("amount_minor_units") * i).alias(f"f{i}") for i in range(20)])
p.select("order_id", "customer", "amount", "channel", "order_date", "is_weekend", "f19").show()

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li><code>withColumn</code> adds a column, or replaces it if the name exists</li><li>Wrap constants in <code>F.lit</code></li><li><code>withColumns({...})</code> adds several at once. They can't reference each other in the same call</li><li><code>withColumnRenamed</code> silently ignores missing columns</li><li>Don't call <code>withColumn</code> hundreds of times in a loop. Use one <code>select</code> or <code>withColumns</code></li><li>The optimizer merges chained projections where it can, and runs them in one code-generated stage, so runtime is fine. The cost is in planning</li><li>Add audit columns (<code>current_timestamp</code>, <code>lit(source)</code>) in Bronze and Silver</li></ul>
</div>

| Task | Code |
|---|---|
| Add / replace | `df.withColumn("x", expr)` |
| Constant | `F.lit("value")` |
| Several | `df.withColumns({"a": e1, "b": e2})` |
| Rename | `df.withColumnRenamed("old", "new")` |
| Rename several | `df.withColumnsRenamed({"o1": "n1"})` |
| Many derived | `df.select("*", *[expr.alias(n) for ...])` |

## Git commit

```
git add 01_spark_basics/07_withcolumn.ipynb
git commit -m "add 07 withcolumn notebook"
```